In [44]:
from bert_score import score
import pandas as pd  
import random
import os
from dotenv import load_dotenv
from pathlib import Path
import mlflow

In [14]:
file_path = '../data/funding_finetune_data.jsonl' 
df = pd.read_json(path_or_buf=file_path, lines=True)

In [15]:
df['prompt'][0], df['completion'][0]

('I need a grant or funding that supports Freshwater biodiversity conservation in Mediterranean Basin (water catchment management)&#8203;:contentReference[oaicite:2]{index=2}. Does Critical Ecosystem Partnership Fund (CEPF) provide this kind of funding?',
 'Critical Ecosystem Partnership Fund (CEPF) offers Up to US$180,000&#8203;:contentReference[oaicite:0]{index=0}. Their application deadline is 11 April 2025&#8203;:contentReference[oaicite:1]{index=1}. Eligibility: Non-governmental organizations, community groups, universities, and private enterprises in the Mediterranean Basin&#8203;:contentReference[oaicite:3]{index=3}. Application process: Submit Letter of Inquiry (LOI) via CEPF website&#8203;:contentReference[oaicite:4]{index=4}.')

In [23]:
txt = 'contentReference'
sum(df['prompt'].apply(lambda x: txt in x)), sum(df['completion'].apply(lambda x: txt in x))

(68, 173)

In [24]:
txt= '&#'
sum(df['prompt'].apply(lambda x: txt in x)), sum(df['completion'].apply(lambda x: txt in x))

(68, 173)

-----

In [ ]:
dotenv_path = Path('/Users/sanchaynibagade/Documents/github/fundraising-ai/env/llm.env')
load_dotenv(dotenv_path=dotenv_path)

MY_KEY = os.getenv('MISTRAL_KEY')
os.environ["MISTRAL_API_KEY"] = MY_KEY

In [26]:

correctness_example_score_1 = mlflow.metrics.genai.EvaluationExample(
    input="What is the capital of France?",
    output=(
        "The capital of France is Paris."
    ),
    score=1,
    justification=(
        "The response is factually correct as Paris is indeed the capital of France"
    ),
)
correctness_example_score_0 = mlflow.metrics.genai.EvaluationExample(
    input="What is the chemical formula of water?",
    output=(
        "The chemical formula of water is H3O",
    ),
    score=0,
    justification=("The chemical formula of water is H20 and not H30, thus the answer is factually incorrect"),
)

In [27]:
correctness = mlflow.metrics.genai.make_genai_metric(
    name="Correctness",
    definition=(
        "Correctness is a binary evaluation metric that determines whether a model-generated answer to a given question is \
        factually accurate. It returns: 1 if the answer is factually correct and directly answers the question.\
        0 if the answer is factually incorrect, misleading, irrelevant, or incomplete"
    ),
    grading_prompt=(
        
        "- Score 0: If the answer is factually incorrect, misleading, irrelevant, or incomplete"
        "- Score 1: If the answer is factually correct and directly answers the question "
    
    ),
    examples=[correctness_example_score_0, correctness_example_score_1],
    model="mistral:/mistral-small-latest",
    parameters={"temperature": 0.0},
    aggregations=["mean", "variance"],
    greater_is_better=True,
)

In [ ]:
df['prompt'][0], df['completion'][0]

In [28]:
res = correctness(  inputs=df['prompt'][0],
                    predictions=df['completion'][0]
                )

100%|██████████| 1/1 [00:01<00:00,  1.07s/it]


In [41]:
res

MetricValue(scores=[0], justifications=['The output states that Neil Armstrong wrote Romeo and Juliet, which is factually incorrect. The correct answer is William Shakespeare.'], aggregate_results={'mean': np.float64(0.0), 'variance': np.float64(0.0)})

In [34]:
## Testing
q = "Who wrote the play Romeo and Juliet?"
a = "Neil Armstrong wrote Romeo and Juliet."

In [32]:
res = correctness(  inputs=q,
                    predictions=a
                )

100%|██████████| 1/1 [00:00<00:00,  1.67it/s]


## Running it for the promts

In [45]:
df['correctness_score'] = ''
df['justifications'] = ''

In [46]:
while sum(df['correctness_score'] == '') != 0:

    idxs = list(df[df['correctness_score'] == ''].index)
    picked_idxs = random.choices(idxs, k=30)
    for i in picked_idxs:
        q,a = df.loc[i,'prompt'], df.loc[i,'completion']
        res = correctness(  inputs=q,
                            predictions=a
                        )
        if res.scores[0] != None:
                df.loc[i,'correctness_score'] = res.scores[0]
                df.loc[i,'justifications'] = res.justifications[0]

100%|██████████| 1/1 [00:00<00:00,  1.71it/s]
/opt/anaconda3/envs/llm/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/opt/anaconda3/envs/llm/lib/python3.13/site-packages/numpy/_core/_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/opt/anaconda3/envs/llm/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:4268: RuntimeWarning: Degrees of freedom <= 0 for slice
  return _methods._var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/opt/anaconda3/envs/llm/lib/python3.13/site-packages/numpy/_core/_methods.py:181: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/opt/anaconda3/envs/llm/lib/python3.13/site-packages/numpy/_core/_methods.py:215: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
100%|██████████| 1/1 [00:00<00

('I need a grant or funding that supports Freshwater biodiversity conservation in Mediterranean Basin (water catchment management)&#8203;:contentReference[oaicite:2]{index=2}. Does Critical Ecosystem Partnership Fund (CEPF) provide this kind of funding?',
 'Critical Ecosystem Partnership Fund (CEPF) offers Up to US$180,000&#8203;:contentReference[oaicite:0]{index=0}. Their application deadline is 11 April 2025&#8203;:contentReference[oaicite:1]{index=1}. Eligibility: Non-governmental organizations, community groups, universities, and private enterprises in the Mediterranean Basin&#8203;:contentReference[oaicite:3]{index=3}. Application process: Submit Letter of Inquiry (LOI) via CEPF website&#8203;:contentReference[oaicite:4]{index=4}.')

In [48]:
df['correctness_score'].value_counts()

correctness_score
1    189
0    163
Name: count, dtype: int64

In [61]:
check_df = df[df['correctness_score'] == 0].reset_index(drop=True)
check_df.loc[0,'prompt'],check_df.loc[0,'completion'],check_df.loc[0,'justifications']

('Are there any special conditions or gotchas when applying to Critical Ecosystem Partnership Fund (CEPF)?',
 'For Critical Ecosystem Partnership Fund (CEPF), gotchas include: LOI required; must align with Mediterranean Basin ecosystem profile; proposals accepted in English, French, or Arabic&#8203;:contentReference[oaicite:6]{index=6}. Additional notes: CEPF is a global partnership fund supporting biodiversity hotspots.',
 'The output mentions "gotchas" but does not directly address whether there are any special conditions or gotchas when applying to CEPF. The information provided is relevant but does not directly answer the question.')

In [63]:
check_df.loc[1,'prompt'],check_df.loc[1,'completion'],check_df.loc[1,'justifications']

('Any past recipients for Critical Ecosystem Partnership Fund (CEPF)?',
 'Past recipients include: Not specified (CEPF has funded many local NGOs in hotspot regions). For more info, contact www.cepf.net&#8203;:contentReference[oaicite:5]{index=5}.',
 'The output does not provide specific past recipients of the Critical Ecosystem Partnership Fund (CEPF), making the answer incomplete and not directly answering the question.')

In [64]:
idx = 2
check_df.loc[idx,'prompt'],check_df.loc[idx,'completion'],check_df.loc[idx,'justifications']

('Does The Iris Project support projects in Worldwide?',
 'The Iris Project focuses on Worldwide. Feel free to reach out via theirisproject.org&#8203;:contentReference[oaicite:12]{index=12} for more details.',
 'The output does not directly answer the question about whether The Iris Project supports projects worldwide. It only mentions that the project focuses on Worldwide, which is not the same as supporting projects.')

In [71]:
idx = 69
check_df.loc[idx,'prompt'],check_df.loc[idx,'completion'],check_df.loc[idx,'justifications']

('Does Oak Spring Garden Foundation (OSGF) support projects in Worldwide?',
 'Oak Spring Garden Foundation (OSGF) focuses on Worldwide. Feel free to reach out via osgf.org (Interdisciplinary Residency)&#8203;:contentReference[oaicite:217]{index=217} for more details.',
 'The output does not directly answer the question about whether OSGF supports projects worldwide. It mentions that OSGF focuses on worldwide but does not confirm support for projects, making the answer incomplete.')

In [66]:
df.to_csv('../data/finetning_data_assessment.csv')